In [ ]:
# python 3.8.8

In [1]:

import cv2 
import mediapipe as mp 
import pygame 
import time 

pygame.mixer.init()


min_play_time=2.0

#light Purple paleete (BGR)
normal_key=(255,233,240)
normal_text=(250,139,167)

active_key=(166,66,104)
active_text=(255,255,255)

playing_color=(250,139,167)
ready_color=(250,139,167)

hand_color=(250,139,167)
conection_color=(196,140,124)

#dark purple black keys(BGR)
black_key=(85,45,65)

mp_hands=mp.solutions.hands
hands=mp_hands.Hands(
  max_num_hands=2,
  min_detection_confidence=0.5,
  min_tracking_confidence=0.5
)

mp_draw=mp.solutions.drawing_utils

hand_style=mp_draw.DrawingSpec(
  color=hand_color,
  circle_radius=5,
  thickness=1
)

conection_style=mp_draw.DrawingSpec(
  color=conection_color,
  thickness=1
)

note_files={
  "C":"ogg/INSPiano_Vel01_C4_01.ogg",
  "D":"ogg/INSPiano_Vel01_D4_01.ogg",
  "E":"ogg/INSPiano_Vel01_E4_01.ogg",
  "F":"ogg/INSPiano_Vel01_F4_01.ogg",
  "G":"ogg/INSPiano_Vel01_G4_01.ogg",
  "A":"ogg/INSPiano_Vel01_A4_01.ogg",
  "B":"ogg/INSPiano_Vel01_B4_01.ogg"
}

note_sounds={
  name:pygame.mixer.Sound(path)
  for name,path in note_files.items()
}

note_channels={
  name: pygame.mixer.Channel(i)
  for i,name in enumerate(note_files)
}

active_notes={}
played_notes={}

#white key positions
key_positions={
  "C":(0,0,150,253),
  "D":(152,0,298,248),
  "E":(300,0,446,253),
  "F":(448,0,594,248),
  "G":(596,0,742,253),
  "A":(744,0,890,248),
  "B":(892,0,1038,253)
}

#black key positions 
black_key_positions={
  "C#":(122,0,190,155),
  "D#":(260,0,338,155),
  "F#":(408,0,486,155),
  "G#":(556,0,634,155),
  "A#":(704,0,782,155)
}

cap=cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1040)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,760)

try:
  while True:
    ret,frame=cap.read()
    if not ret :
      break

    frame=cv2.flip(frame,1)
    h,w,_=frame.shape

    frame_rgb=cv2.cvtColor(
      frame,cv2.COLOR_BGR2RGB
    )

    results=hands.process(frame_rgb)
    current_notes={}

    if results.multi_hand_landmarks:
      for i , hand_landmarks in enumerate(
        results.multi_hand_landmarks
      ):
        index_finger=hand_landmarks.landmark[8]
        x=int(index_finger.x*w)
        y=int(index_finger.y*h)

        current_note=None

        #detect black keys first 
        for note,(x1,y1,x2,y2)in black_key_positions.items():
          if x1<x<x2 and y1<y<y2:
            current_note=note
            break 
        #detect white keys if no black keys was detected
        if current_note is None: 
            if 0<x<150 and 0<y<253:
              current_note ="C"
            elif 152<x<298 and 0<y<248:
              current_note="D"
            elif 300<x<446 and 0< y <253:
              current_note="E"
            elif 448<x<594 and 0< y <248:
              current_note= "F"
            elif 596<x<742 and 0<y<253:
              current_note="G"
            elif 744<x<890 and 0<y<248:
              current_note="A"
            elif 892<x<1038 and 0<y<253:
              current_note = "B"
        if current_note is not None:
            current_notes[i]=current_note

        mp_draw.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            hand_style,
            conection_style
          )
      #draw white keys
    for note,(x1,y1,x2,y2)in key_positions.items():
        is_active=note in current_notes.values()
        color=active_key if is_active else normal_key
        text_color=active_text if is_active else normal_text

        cv2.rectangle(
          frame,
          (x1,y1),
          (x2,y2),
          color,
          -1
        )

        cv2.putText(
          frame,
          note,
          (x1+(x2-x1)//2-12,y2-18),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.8,
          text_color,
          2,
          cv2.LINE_AA
        )

      #draw black keys on top 
    for note,(x1,y1,x2,y2)in black_key_positions.items():
        cv2.rectangle(
          frame,
          (x1,y1),
          (x2,y2),
          black_key,
          -1
        )

    now=time.monotonic()
    for hand_id,note_name in current_notes.items():
        previous_note=active_notes.get(hand_id)

        if note_name != previous_note:
          if note_name in note_sounds:
            note_channels[note_name].play(
              note_sounds[note_name]
            )

          played_notes[hand_id]=(note_name,now)
      
    for hand_id,(note_name,start_time)in list(
        played_notes.items()
      ):
        released=(
          hand_id not in current_notes
          or current_notes[hand_id] != note_name
        )

        elapsed=now - start_time 

        if released and elapsed >= min_play_time :
          played_notes.pop(hand_id,None)
        
    active_notes=current_notes.copy()

    status_text=(
        f"PLAYING:{', '.join(current_notes.values())}"
        if current_notes
        else "READY TO PLAY"
      )

    status_color=(
        playing_color if current_notes else ready_color
      )

    cv2.putText(
        frame,
        status_text,
        (24,280),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        status_color,
        2,
        cv2.LINE_AA
      )

    cv2.imshow("PIANO",frame)
    if cv2.waitKey(1) == ord('q'):
        break
finally:
  cap.release()
  hands.close()
  pygame.quit()
  cv2.destroyAllWindows()


pygame 2.6.1 (SDL 2.28.4, Python 3.8.8)
Hello from the pygame community. https://www.pygame.org/contribute.html
